# 유형 07 — 동적계획법 (DP, Dynamic Programming)

> **대상**: 완전탐색(재귀)과 그리디를 겪은 단계.
> **목표**: "큰 문제를 작은 문제로 쪼개고, 그 답을 저장해 재사용"하는 DP의 사고법 — 특히 **점화식 세우기** — 을 익히기.

DP는 코테에서 가장 막막하게 느껴지는 유형입니다. 하지만 본질은 단순합니다: **"같은 작은 문제를 여러 번 풀게 되니, 한 번 푼 답을 저장(memo)해서 재사용한다."** 완전탐색이 너무 느릴 때(겹치는 부분 문제가 많을 때) DP로 바꿉니다.

## 0) DP를 의심하는 신호

| 신호 | 설명 |
|------|------|
| "경우의 수 / 가짓수를 구하라" | 작은 경우의 합으로 표현되는 경우가 많음 |
| "최소/최대 비용, 최장/최단 ~" | 작은 선택의 최적을 쌓아 큰 최적 |
| 완전탐색이 시간초과 | 겹치는 부분 문제 → memo로 가속 |
| "이전 상태에 의존" | `dp[i]`가 `dp[i-1]`, `dp[i-2]`... 로 정의됨 |

> 그리디(04)와 헷갈릴 때: **그리디가 반례로 깨지면 DP**입니다. DP는 "모든 선택지를 고려하되 중복 계산만 없앤" 것이라 항상 최적을 보장합니다.

## 1) 핵심 — 점화식 세우기 (DP의 전부)

DP의 90%는 **"`dp[i]`를 한 문장으로 정의하고, `dp[i]`를 더 작은 `dp`로 표현하기"**입니다.

**예: 계단 오르기 (한 번에 1칸 또는 2칸)**
- `dp[i]` = "i번째 계단에 도달하는 경우의 수"
- i칸에 오는 방법은? **(i-1)칸에서 1칸** 또는 **(i-2)칸에서 2칸** → `dp[i] = dp[i-1] + dp[i-2]`
- 시작값(base): `dp[1] = 1`, `dp[2] = 2`

```python
def climb(n):
    if n <= 2:
        return n
    dp = [0] * (n + 1)
    dp[1], dp[2] = 1, 2
    for i in range(3, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]   # 점화식
    return dp[n]

climb(5)   # 8
```

> 점화식 세우는 3단계:
> 1. **`dp[i]`가 무엇인지** 한국어로 정의한다 ("i까지의 최대 이익" 등).
> 2. **`dp[i]`를 더 작은 `dp`로** 표현한다 (마지막 선택만 생각).
> 3. **base case**(가장 작은 값)를 정한다.

In [1]:
def climb(n):
    if n <= 2:
        return n
    dp = [0] * (n + 1)
    dp[1], dp[2] = 1, 2
    for i in range(3, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

climb(5)

8

## 2) 두 가지 구현 방식

같은 점화식을 **위에서 아래로(재귀+memo)** 또는 **아래에서 위로(반복+표)** 구현합니다.

**(A) 메모이제이션 — 재귀 + 캐시 (Top-down)**

```python
from functools import lru_cache

@lru_cache(maxsize=None)     # 한 번 계산한 결과를 자동 저장
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

fib(10)   # 55
```

> `@lru_cache`만 붙이면 완전탐색 재귀가 곧 DP가 됩니다. "재귀로 풀리는데 느리면" 가장 먼저 시도.

**(B) 타뷸레이션 — 반복 + 배열 (Bottom-up)**

```python
def fib(n):
    if n <= 1:
        return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]
```

> 둘은 같은 점화식, 다른 방향. **메모이제이션이 떠올리기 쉽고, 타뷸레이션이 빠르고 안전**(재귀 깊이 제한 없음)합니다.

In [2]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

fib(10)

55

In [3]:
def fib(n):
    if n <= 1:
        return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

fib(10)

55

## 대표 문제 풀이 — N으로 표현 (프로그래머스 42895)

> 숫자 `N`을 5개 이하로 사용해 사칙연산으로 `number`를 만들 때, **사용한 N의 최소 개수**를 구하라. 못 만들면 -1. (예: N=5로 12 만들기 → `55/5+5/5 = 12`, 4개)

**① 신호**: "최소 개수", "작은 경우를 조합" — N을 k개 쓴 결과는 (i개 결과)와 (k-i개 결과)의 연산
**② 패턴**: DP — `dp[k]` = "N을 k개 써서 만들 수 있는 모든 수의 집합"

```python
def solution(N, number):
    if N == number:
        return 1
    # dp[k] = N을 k개 사용해 만들 수 있는 수들의 집합
    dp = [set() for _ in range(9)]   # 1~8개
    for k in range(1, 9):
        # NN, NNN ... (N을 k번 이어 붙인 수)
        dp[k].add(int(str(N) * k))
        # i개 결과와 (k-i)개 결과를 사칙연산으로 조합
        for i in range(1, k):
            for a in dp[i]:
                for b in dp[k - i]:
                    dp[k].add(a + b)
                    dp[k].add(a - b)
                    dp[k].add(a * b)
                    if b != 0:
                        dp[k].add(a // b)
        if number in dp[k]:
            return k
    return -1
```

**검증:**

```python
print(solution(5, 12))    # 4   (55/5 + 5/5)
print(solution(2, 11))    # 3   (22/2)
print(solution(5, 0))     # 2   (5 - 5)

assert solution(5, 12) == 4
assert solution(2, 11) == 3
assert solution(5, 0) == 2
print("통과 ✅")
```

> 가르칠 포인트:
> 1. **`dp[k]`가 "값" 하나가 아니라 "집합"** — N을 k개 써서 나올 수 있는 모든 수를 모읍니다.
> 2. **조합 점화식** — `dp[k]`는 `dp[i]`와 `dp[k-i]`의 모든 쌍을 사칙연산한 결과 + N을 k번 이어붙인 수(`NN`, `NNN`).
> 3. **작은 k부터** 차곡차곡 쌓아, `number`가 처음 등장하는 k가 최소 개수. 8개까지 못 만들면 -1.

In [5]:
def solution(N, number):
    if N == number:
        return 1
    dp = [set() for _ in range(9)]
    for k in range(1, 9):
        dp[k].add(int(str(N) * k))
        for i in range(1, k):
            for a in dp[i]:
                for b in dp[k - i]:
                    dp[k].add(a + b)
                    dp[k].add(a - b)
                    dp[k].add(a * b)
                    if b != 0:
                        dp[k].add(a // b)
        if number in dp[k]:
            return k
    return -1

In [6]:
print(solution(5, 12))    # 4   (55/5 + 5/5)
print(solution(2, 11))    # 3   (22/2)
print(solution(5, 0))     # 2   (5 - 5)

assert solution(5, 12) == 4
assert solution(2, 11) == 3
assert solution(5, 0) == 2
print("통과 ✅")

4
3
2
통과 ✅


## 직접 풀어보기 (연습)

힌트를 보고 `TODO`를 채운 뒤 검증 셀을 실행하세요. DP는 **"`dp[i]`를 정의하고 점화식을 세우는 것"**이 9할입니다.


### 연습 1 — 1로 만들기 (점화식 기본)

> 정수 `x`에 (1) `-1`, (2) `/2`(2의 배수일 때), (3) `/3`(3의 배수일 때) 연산을 써서 1로 만들 때, **최소 연산 횟수**를 구하라.
> **힌트**: `dp[i]` = i를 1로 만드는 최소 횟수. `dp[i] = dp[i-1] + 1`을 기본으로, i가 2·3의 배수면 `dp[i//2]+1`, `dp[i//3]+1`과 비교해 최솟값.

```python
def solution(x):
    # TODO: dp 배열로 직접 작성
    pass

# 검증
assert solution(26) == 5    # 26→13(/2)→12(-1)→4(/3)→3(-1)→1(/3)
assert solution(10) == 3    # 10→9(-1)→3(/3)→1(/3)
assert solution(1) == 0
print("통과 ✅")
```

### 연습 2 — 정수 삼각형 (프로그래머스 43105)

> 삼각형 `triangle`의 꼭대기에서 바닥까지 내려가며 거치는 수의 **합의 최댓값**을 구하라. 아래로 내려갈 때 바로 아래 또는 오른쪽 아래로만 이동.
> **힌트**: `dp[r][c]` = (r,c)까지 오는 최대 합 = `triangle[r][c] + max(dp[r-1][c-1], dp[r-1][c])`. 가장자리 인덱스 주의.

```python
def solution(triangle):
    # TODO: 직접 작성
    pass

# 검증
assert solution([[7], [3, 8], [8, 1, 0], [2, 7, 4, 4], [4, 5, 2, 6, 5]]) == 30
assert solution([[1], [2, 3]]) == 4
print("통과 ✅")
```

### 연습 3 — 도둑질(원형 집) 간소화 버전 (도전)

> 일렬로 늘어선 집의 금액 리스트 `money`. **인접한 두 집은 동시에 털 수 없을 때** 훔칠 수 있는 최대 금액을 구하라. (원형 아님, 일렬 버전)
> **힌트**: `dp[i]` = i번째 집까지 고려한 최대 금액 = `max(dp[i-1], dp[i-2] + money[i])`. (i를 안 털거나 / 털고 i-2까지 더하거나)

```python
def solution(money):
    # TODO: 직접 작성
    pass

# 검증
assert solution([1, 2, 3, 1]) == 4      # 1 + 3
assert solution([2, 7, 9, 3, 1]) == 12  # 2 + 9 + 1
assert solution([5]) == 5
print("통과 ✅")
```